# Qwen-Image-2.1 — ComfyUI INT8, custom prompt version 2

This version keeps the confirmed ComfyUI INT8 setup but replaces the official workflow prompt with a custom still-life prompt. The previous fashion prompt and PNG remain in the version history of this kernel.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

from urllib.request import Request, urlopen

WORKING = Path('/kaggle/working')
COMFY_ROOT = WORKING / 'ComfyUI'
COMFY_OUTPUT = WORKING / 'comfy_output'
RESULT_ROOT = WORKING / 'qwen_image_2_1_result'
WORKFLOW_PATH = WORKING / 'qwen_image_2_1_t2i.json'
CUSTOM_PROMPT = 'A cinematic editorial still life of a weathered brass compass resting on an open antique book, beside a small glass jar of dried botanical specimens and a folded vintage map. Soft morning sunlight enters from the left through a window, creating delicate dust motes, warm copper highlights, and long quiet shadows across the table. The compass needle points northeast. Fine paper fibers, engraved metal, tiny cracks in the glass, and subtle botanical textures are rendered with realistic detail. The composition is balanced, tactile, and nostalgic, with a muted palette of amber, cream, forest green, and charcoal. Professional natural-light photography, 50mm lens, shallow depth of field, high dynamic range, realistic materials, clean composition, no people, no text, no lettering, no logo, no watermark.'
SERVER_LOG = WORKING / 'comfy_server.log'
REPORT_PATH = WORKING / 'qwen_image_2_1_report.json'

COMFY_COMMIT = '93810483a4739a1588236919a3128d3070244146'
WORKFLOW_COMMIT = 'a7acaf8cee9ccccdaf4b26ce04a9fced4c4d13de'
WORKFLOW_URL = (
    'https://raw.githubusercontent.com/Comfy-Org/workflow_templates/'
    f'{WORKFLOW_COMMIT}/templates/image_qwen_image_2_1_t2i.json'
)
MODEL_REPO = 'Comfy-Org/Qwen-Image-2.1'
MODEL_FILES = {
    'diffusion_models': 'diffusion_models/qwen_image_2.1_int8_convrot.safetensors',
    'text_encoders': 'text_encoders/qwen3vl_8b_int8_convrot.safetensors',
    'vae': 'vae/qwen_image_2.1_vae_bf16.safetensors',
}

for path in (COMFY_OUTPUT, RESULT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

report = {
    'stage': 'starting',
    'comfy_commit': COMFY_COMMIT,
    'workflow_commit': WORKFLOW_COMMIT,
    'model_repo': MODEL_REPO,
    'model_files': MODEL_FILES,
    'success': False,
}

def save_report():
    REPORT_PATH.write_text(json.dumps(report, indent=2, default=str) + '\n')
    print(json.dumps(report, indent=2, default=str))

def run(command, cwd=None, timeout=None):
    print('+', ' '.join(map(str, command)), flush=True)
    return subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )

def check_run(result, label):
    if result.returncode != 0:
        report['stage'] = f'failed: {label}'
        report['last_command_returncode'] = result.returncode
        report['last_command_stdout'] = result.stdout[-12000:]
        report['last_command_stderr'] = result.stderr[-12000:]
        save_report()
        raise RuntimeError(f'{label} failed with exit code {result.returncode}')

save_report()


In [ ]:
if not COMFY_ROOT.exists():
    result = run([
        'git', 'init', str(COMFY_ROOT),
    ])
    check_run(result, 'git init')
    result = run(['git', '-C', str(COMFY_ROOT), 'remote', 'add', 'origin', 'https://github.com/Comfy-Org/ComfyUI.git'])
    check_run(result, 'git remote add')
    result = run([
        'git', '-C', str(COMFY_ROOT), 'fetch', '--depth', '1', 'origin', COMFY_COMMIT,
    ], timeout=600)
    check_run(result, 'git fetch ComfyUI commit')
    result = run(['git', '-C', str(COMFY_ROOT), 'checkout', '--detach', 'FETCH_HEAD'])
    check_run(result, 'git checkout ComfyUI commit')

report['stage'] = 'installing ComfyUI dependencies'
save_report()
result = run([
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-input',
    '-r', str(COMFY_ROOT / 'requirements.txt'), 'comfy-cli', 'huggingface_hub',
], timeout=1800)
check_run(result, 'pip install ComfyUI dependencies')
report['comfyui_path'] = str(COMFY_ROOT)
report['python'] = sys.executable
save_report()


In [ ]:
from huggingface_hub import hf_hub_download

report['stage'] = 'downloading official workflow and INT8 weights'
save_report()
request = Request(WORKFLOW_URL, headers={'User-Agent': 'kaggle-qwen-image-test/1.0'})
with urlopen(request, timeout=120) as response:
    workflow_bytes = response.read()
WORKFLOW_PATH.write_bytes(workflow_bytes)
workflow = json.loads(workflow_bytes)
prompt_replacements = 0

def replace_custom_prompt(value):
    global prompt_replacements
    if isinstance(value, dict):
        named = value.get('widgets_values_named')
        values = value.get('widgets_values')
        if isinstance(named, dict) and 'prompt' in named:
            named['prompt'] = CUSTOM_PROMPT
            if isinstance(values, list) and values and isinstance(values[0], str):
                values[0] = CUSTOM_PROMPT
            prompt_replacements += 1
        for child in value.values():
            replace_custom_prompt(child)
    elif isinstance(value, list):
        for child in value:
            replace_custom_prompt(child)

replace_custom_prompt(workflow)
if prompt_replacements != 1:
    raise RuntimeError(f'Expected one prompt replacement, got {prompt_replacements}')
report['custom_prompt'] = CUSTOM_PROMPT
report['prompt_replacements'] = prompt_replacements
report['workflow_bytes'] = len(workflow_bytes)
report['workflow_has_nodes'] = isinstance(workflow.get('nodes'), list)
report['workflow_node_count'] = len(workflow.get('nodes', []))

for model_folder, filename in MODEL_FILES.items():
    destination = COMFY_ROOT / 'models' / model_folder
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / Path(filename).name
    if target.exists() and target.stat().st_size > 0:
        print('already present:', target)
        continue
    downloaded = hf_hub_download(
        repo_id=MODEL_REPO,
        filename=filename,
        local_dir=str(COMFY_ROOT / 'models'),
        local_dir_use_symlinks=False,
    )
    downloaded_path = Path(downloaded)
    if downloaded_path != target:
        shutil.copy2(downloaded_path, target)
    print('downloaded:', target, target.stat().st_size)

report['downloaded_model_sizes'] = {
    folder: (COMFY_ROOT / 'models' / folder / Path(filename).name).stat().st_size
    for folder, filename in MODEL_FILES.items()
}
report['stage'] = 'starting ComfyUI server'
save_report()


In [ ]:
server = None
run_result = None
start_time = time.time()
try:
    server_log_handle = SERVER_LOG.open('w', encoding='utf-8')
    server = subprocess.Popen(
        [
            sys.executable, 'main.py',
            '--listen', '127.0.0.1',
            '--port', '8188',
            '--disable-auto-launch',
            '--lowvram',
            '--force-fp16',
            '--output-directory', str(COMFY_OUTPUT),
        ],
        cwd=str(COMFY_ROOT),
        stdout=server_log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )

    server_ready = False
    for _ in range(120):
        if server.poll() is not None:
            break
        try:
            with urlopen('http://127.0.0.1:8188/system_stats', timeout=3) as response:
                if response.status == 200:
                    server_ready = True
                    break
        except Exception:
            time.sleep(2)
    if not server_ready:
        report['server_log_tail'] = SERVER_LOG.read_text(errors='replace')[-16000:]
        save_report()
        raise RuntimeError('ComfyUI server did not become ready')

    report['stage'] = 'running official Qwen-Image-2.1 workflow'
    report['workflow_path'] = str(WORKFLOW_PATH)
    save_report()
    run_result = run([
        'comfy', 'run',
        '--workflow', str(WORKFLOW_PATH),
        '--where', 'local',
        '--host', '127.0.0.1',
        '--port', '8188',
        '--wait',
        '--timeout', '1800',
    ], timeout=1900)
    report['comfy_run_returncode'] = run_result.returncode
    report['comfy_run_stdout_tail'] = run_result.stdout[-12000:]
    report['comfy_run_stderr_tail'] = run_result.stderr[-12000:]
    check_run(run_result, 'comfy run')
finally:
    if server is not None:
        server.terminate()
        try:
            server.wait(timeout=30)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait(timeout=30)
    if 'server_log_handle' in locals():
        server_log_handle.close()


In [ ]:
# Preserve a small, explicit result set for Kaggle output.
copied = []
for source in COMFY_OUTPUT.rglob('*'):
    if source.is_file():
        relative = source.relative_to(COMFY_OUTPUT)
        target = RESULT_ROOT / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
        copied.append(str(target.relative_to(RESULT_ROOT)))

report['output_files'] = copied
report['png_files'] = [name for name in copied if name.lower().endswith('.png')]
report['elapsed_seconds'] = round(time.time() - start_time, 2)
report['server_log_tail'] = SERVER_LOG.read_text(errors='replace')[-16000:] if SERVER_LOG.exists() else ''
report['success'] = bool(report['png_files'])
report['stage'] = 'complete' if report['success'] else 'completed_without_png'
save_report()

if not report['success']:
    raise RuntimeError('ComfyUI completed without a PNG output')
